# 03 · Elo Baseline

**Goal:** Rate every NFL team with Elo and score its pre-game win probabilities against actual results.

**Data:** `nflmodel.data.read_games()`, filtered to `FIRST_SEASON`–`LAST_SEASON`

**Code:** `nflmodel.elo` (math, engine, reshape) and `nflmodel.evaluate` (pick accuracy, Brier)

**Contents:** 0. Setup & parameters · 1. Load · 2. Elo math · 3. Elo engine · 4. Reshape ·
pick accuracy · 5. Brier score · 6. Tune

Builds a baseline Elo rating for every NFL team, updated game by game and regressed toward the mean each offseason. All settings live in Section 0, so the model runs for any range of seasons. The output is a pre-game win probability for every game, scored against actual results.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from nflmodel.data import read_games
from nflmodel.elo import DEFAULT_PARAMS

In [2]:
# =============================
# Section 0: setup & parameters
# =============================

# Timeframe
FIRST_SEASON = 2005
LAST_SEASON = 2025
SCORE_FROM = 2013
TRAIN_SEASONS = range(2013, 2022)
TEST_SEASONS = range(2022, 2026)

# Model parameters (copy so tweaks here don't touch the package defaults)
params = {**DEFAULT_PARAMS}

In [3]:
# =============================
# Section 1: load & clean
# =============================

games = read_games()
games = games[games["season"].between(FIRST_SEASON, LAST_SEASON)].reset_index(drop=True)

### Section 2: Elo math

The building blocks for a single game, in `nflmodel.elo`:

- `elo_diff`: the home team's rating edge, including home-field advantage
- `home_win_prob`: turns that edge into a home win probability
- `predicted_spread`: turns the edge into an expected home margin in points
- `mov_multiplier`: scales the update by margin of victory, damped when the favorite wins
- `rating_change`: Elo points the home team gains (the away team loses the same amount)
- `regress_ratings`: pulls every rating part of the way back toward 1500 between seasons

In [4]:
# =============================
# Section 2: elo math
# =============================

from nflmodel.elo import (elo_diff, home_win_prob, predicted_spread,
                          mov_multiplier, rating_change, regress_ratings)

### Section 3: Elo engine

`run_elo(games, params)` walks every game in kickoff order. At each new season it regresses ratings toward the mean, then for each game it looks up pre-game ratings (new teams start at base), predicts the home win probability, and, if the game has been played, updates both teams. It returns the games with `elo_home_pre`, `elo_away_pre`, `home_prob`, `elo_home_post`, and `elo_away_post` added.

In [5]:
# =============================
# Section 3: elo engine
# =============================

from nflmodel.elo import run_elo

### Section 4: Reshape

- `team_history(elo)`: one row per team per game, with `elo_pre` and `elo_post`
- `ratings_table(history, value)`: wide table with (season, week) rows and one column per team, forward-filled through bye weeks

In [6]:
# =============================
# Section 4: reshape
# =============================

from nflmodel.elo import team_history, ratings_table

In [7]:
elo = run_elo(games, params)
elo.head(10)

,game_id,season,week,kickoff,game_type,is_playoff,home_team,away_team,location,is_neutral,...,home_qb_id,away_qb_id,spread_line,home_moneyline,away_moneyline,elo_home_pre,elo_away_pre,home_prob,elo_home_post,elo_away_post
0,2005_01_OAK_NE,2005,1,2005-09-08 09:00:00,REG,False,NE,LV,Home,False,...,00-0019596,00-0003292,7.5,NaN,NaN,1500.0,1500.0,0.568641,1520.245339,1479.754661
1,2005_01_CHI_WAS,2005,1,2005-09-11 13:00:00,REG,False,WAS,CHI,Home,False,...,00-0021379,00-0023541,6.0,NaN,NaN,1500.0,1500.0,0.568641,1509.275542,1490.724458
2,2005_01_CIN_CLE,2005,1,2005-09-11 13:00:00,REG,False,CLE,CIN,Home,False,...,00-0004293,00-0021429,-3.5,NaN,NaN,1500.0,1500.0,0.568641,1468.514863,1531.485137
3,2005_01_DEN_MIA,2005,1,2005-09-11 13:00:00,REG,False,MIA,DEN,Home,False,...,00-0005589,00-0013042,-4.5,NaN,NaN,1500.0,1500.0,0.568641,1527.176847,1472.823153
4,2005_01_HOU_BUF,2005,1,2005-09-11 13:00:00,REG,False,BUF,HOU,Home,False,...,00-0022912,00-0020608,4.5,NaN,NaN,1500.0,1500.0,0.568641,1523.408861,1476.591139
5,2005_01_NO_CAR,2005,1,2005-09-11 13:00:00,REG,False,CAR,NO,Home,False,...,00-0004161,00-0001823,6.5,NaN,NaN,1500.0,1500.0,0.568641,1483.882253,1516.117747
6,2005_01_NYJ_KC,2005,1,2005-09-11 13:00:00,REG,False,KC,NYJ,Home,False,...,00-0006355,00-0019559,3.0,NaN,NaN,1500.0,1500.0,0.568641,1525.704787,1474.295213
7,2005_01_SEA_JAX,2005,1,2005-09-11 13:00:00,REG,False,JAX,SEA,Home,False,...,00-0022177,00-0007091,3.0,NaN,NaN,1500.0,1500.0,0.568641,1521.655770,1478.344230
8,2005_01_TB_MIN,2005,1,2005-09-11 13:00:00,REG,False,MIN,TB,Home,False,...,00-0003739,00-0006423,6.0,NaN,NaN,1500.0,1500.0,0.568641,1471.109241,1528.890759
9,2005_01_TEN_PIT,2005,1,2005-09-11 13:00:00,REG,False,PIT,TEN,Home,False,...,00-0022924,00-0011024,7.0,NaN,NaN,1500.0,1500.0,0.568641,1528.133676,1471.866324


### Pick accuracy

`check_rate(elo)` keeps games that were played and not tied, and adds a `correct` column that is True when the Elo favorite won.

In [8]:
from nflmodel.evaluate import check_rate

checked = check_rate(elo)

print(checked.groupby("home_team")["correct"].mean())
print(f"\nModel Accuracy: {checked.loc[checked["season"] >= SCORE_FROM, "correct"].mean()}")

home_team
ARI    0.578035
ATL    0.613636
BAL    0.703911
BUF    0.688889
CAR    0.590909
CHI    0.698324
CIN    0.597701
CLE    0.617647
DAL    0.607955
DEN    0.608696
DET    0.722543
GB          0.7
HOU    0.657303
IND    0.651934
JAX    0.632184
KC     0.678947
LA     0.602273
LAC    0.651429
LV     0.602339
MIA    0.604651
MIN    0.645714
NE     0.766497
NO          0.6
NYG    0.616279
NYJ    0.625731
PHI    0.666667
PIT    0.662983
SEA    0.717391
SF     0.659218
TB     0.629213
TEN    0.655172
WAS    0.578035
Name: correct, dtype: Float64

Model Accuracy: 0.6476056338028169


### Section 5: Brier score

- `brier_score(df)`: mean squared error of `home_prob` against `home_win`, over played games
- `brier_baselines(df)`: Brier scores of two naive forecasters on the same games: a coin flip (0.25) and always predicting the home win rate

In [9]:
# =============================
# Section 5: evaluate
# =============================

from nflmodel.evaluate import brier_score, brier_baselines

In [10]:
train = elo[elo["season"].isin(list(TRAIN_SEASONS))]
test = elo[elo["season"].isin(list(TEST_SEASONS))]

for name, df in [("train", train), ("test", test)]:
    b = brier_score(df)
    base = brier_baselines(df)
    bss = 1 - b / base["coin_flip"]
    print(f"{name}: Brier {b:.4f} | home-rate baseline {base["home_rate"]:.4f} | BSS {bss:.1%}")

train: Brier 0.2189 | home-rate baseline 0.2469 | BSS 12.4%
test: Brier 0.2218 | home-rate baseline 0.2471 | BSS 11.3%


In [11]:
# =============================
# Section 6: tune
# =============================

def tune(games, k_values, hfa_values, reg_values):
    """ 
    Training-season Brier score for every parameter combination. 
    
    games:      games DataFrame (full history, burn-in included)
    k_values:   list of K values to try
    hfa_values: list of HFA values to try
    reg_values: list of offseason regression values to try
    """

    rows = []
    for k in k_values:
        for hfa in hfa_values:
            for reg in reg_values:
                trial = {**params, "k": k, "hfa": hfa, "season_regression": reg}

                elo_t = run_elo(games, trial)
                train_t = elo_t[elo_t["season"].isin(list(TRAIN_SEASONS))]
                brier = brier_score(train_t)

                rows.append({"k":k, "hfa": hfa, "season_regression": reg, "brier": brier})
    results = pd.DataFrame(rows).sort_values("brier", ascending=True).reset_index(drop=True)
    return results

results = tune(games, 
               k_values = list(range(15, 34, 3)),
               hfa_values = list(range(24, 60, 2)),
               reg_values = [0, .1, .2, .3, .4, .5, .6, .7, .8, .9])
type(results)


pandas.DataFrame

In [12]:
print(results.head(10))
print(results.groupby("hfa")["brier"].min())
print(results.groupby("k")["brier"].min())
print(results.groupby("season_regression")["brier"].min())

    k  hfa  season_regression     brier
0  21   42                0.4  0.218781
1  21   40                0.4  0.218782
2  21   44                0.4  0.218794
3  21   38                0.4  0.218798
4  18   40                0.4  0.218817
5  18   42                0.4  0.218821
6  21   46                0.4  0.218821
7  21   36                0.4  0.218829
8  18   38                0.4  0.218829
9  18   44                0.4  0.218839
hfa
24    0.219322
26    0.219203
28    0.219098
30    0.219009
32    0.218934
34    0.218874
36    0.218829
38    0.218798
40    0.218782
42    0.218781
44    0.218794
46    0.218821
48    0.218863
50    0.218920
52    0.218990
54    0.219075
56    0.219174
58    0.219286
Name: brier, dtype: float64
k
15    0.219254
18    0.218817
21    0.218781
24    0.219062
27    0.219586
30    0.220217
33    0.221003
Name: brier, dtype: float64
season_regression
0.0    0.223722
0.1    0.221207
0.2    0.219692
0.3    0.218948
0.4    0.218781
0.5    0.219012
0.6    0.